# Hypothesis Testing in Python — Practical Jupyter Notebook

A hands-on guide to hypothesis testing for Data Science and Machine Learning.

We will use simple analogies, mathematical intuition, Python code, generated datasets, and interpretation of statistical results.

> **Core idea:** Hypothesis testing asks whether the evidence in our sample is strong enough to reject a default assumption about a population.

In [ ]:
# Install these if needed in a fresh environment:
# %pip install numpy pandas scipy statsmodels matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

np.random.seed(42)

## 1. Population vs Sample

### Analogy: tasting soup 🍲

Imagine a huge pot of soup.

- **Population** = the entire pot
- **Sample** = the spoonful you taste
- **Population parameter** = the true property of the whole pot
- **Sample statistic** = what you calculate from the spoonful

In Data Science, we usually cannot collect data from an entire population, so we use a sample to learn about it.

In [ ]:
population = np.random.normal(loc=70, scale=10, size=10000)

print("Population mean:", population.mean())
print("Population standard deviation:", population.std())

sample = np.random.choice(population, size=100, replace=False)

print("\nSample mean:", sample.mean())
print("Sample standard deviation:", sample.std(ddof=1))

## 2. Sampling Distribution

A **sampling distribution** is the distribution of a statistic, such as the sample mean, obtained from many repeated samples.

### Analogy

Instead of tasting one spoonful of soup, imagine taking **1,000 spoonfuls**, calculating the average taste of each spoonful, and looking at all those averages.

The distribution of those averages is the **sampling distribution of the mean**.

In [ ]:
sample_means = []

for _ in range(1000):
    sample = np.random.choice(population, size=30, replace=False)
    sample_means.append(sample.mean())

plt.hist(sample_means, bins=30)
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Sampling Distribution of the Mean")
plt.show()

## 3. Central Limit Theorem

The **Central Limit Theorem (CLT)** says that, under broad conditions, the distribution of sample means becomes approximately normal as sample size becomes sufficiently large, even when the original population is not normal.

### Analogy

The original population can be messy or skewed. But when we repeatedly take reasonably large samples and calculate their means, those means tend to form a bell-shaped distribution.

In [ ]:
skewed_population = np.random.exponential(scale=10, size=100000)

plt.hist(skewed_population, bins=50)
plt.title("Original Exponential Population")
plt.show()

sample_means = []

for _ in range(5000):
    sample = np.random.choice(skewed_population, size=50)
    sample_means.append(sample.mean())

plt.hist(sample_means, bins=40)
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Sampling Distribution of Means")
plt.show()

# 4. Hypothesis Testing

## The courtroom analogy ⚖️

Imagine a court case.

The defendant starts as **presumed innocent**. That is similar to the **null hypothesis (H₀)**.

The prosecution presents evidence. We ask:

> If the defendant really were innocent, how unusual would this evidence be?

If the evidence is sufficiently strong, we reject the initial assumption.

### Statistical translation

- **H₀ (null hypothesis)** → default assumption, usually "no effect" or "no difference"
- **H₁ / Hₐ (alternative hypothesis)** → what we are looking for evidence for
- **Sample data** → evidence
- **p-value** → how unusual the evidence is if H₀ is true
- **α (alpha)** → our significance threshold
- **Decision** → reject H₀ or fail to reject H₀

### Important language

We normally say:

> **Reject H₀**

or

> **Fail to reject H₀**

We do not normally say that a non-significant test "proves H₀ is true." 

## 5. Significance Level (α)

A common choice is:

\[
\alpha = 0.05
\]

Decision rule:

\[
p < \alpha \Rightarrow \text{Reject H₀}
\]

\[
p \geq \alpha \Rightarrow \text{Fail to reject H₀}
\]

The 0.05 threshold is a convention, not a magical boundary between truth and falsehood.

In [ ]:
alpha = 0.05
print("Significance level:", alpha)

## 6. P-value

The p-value is approximately:

> **The probability, assuming H₀ is true, of obtaining the observed result or something more extreme.**

It is **not** the probability that H₀ is true.

### Intuition

If H₀ predicts that something should be common, but our result is extremely unusual under H₀, the p-value becomes small.

- Small p-value → evidence against H₀
- Large p-value → not enough evidence to reject H₀

# 7. One-Sample t-Test

### Problem

A company claims that the average delivery time is **10 days**.

We collect a sample and want to know whether the population mean is different from 10.

### Hypotheses

\[
H_0: \mu = 10
\]

\[
H_1: \mu \neq 10
\]

Because the population standard deviation is unknown, a one-sample **t-test** is appropriate.

In [ ]:
delivery_times = np.array([
    9.2, 10.1, 9.5, 10.4, 9.8,
    9.7, 10.2, 9.1, 9.6, 10.0
])

result = stats.ttest_1samp(delivery_times, popmean=10)

print("Sample mean:", delivery_times.mean())
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)

if result.pvalue < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")

### Interpretation

If p < 0.05:

> There is statistically significant evidence that the average delivery time differs from 10 days.

If p ≥ 0.05:

> There is insufficient evidence to conclude that the average delivery time differs from 10 days.

**Fail to reject H₀ does not prove H₀.**

# 8. One-Tailed vs Two-Tailed Tests

If we specifically want to know whether delivery time has **decreased**:

\[
H_1: \mu < 10
\]

This is a **one-tailed test**.

If we ask whether it is simply different:

\[
H_1: \mu \neq 10
\]

we use a **two-tailed test**.

In SciPy:

- `alternative="less"` → μ < hypothesized value
- `alternative="greater"` → μ > hypothesized value
- `alternative="two-sided"` → μ ≠ hypothesized value

In [ ]:
less_result = stats.ttest_1samp(
    delivery_times, popmean=10, alternative="less"
)

greater_result = stats.ttest_1samp(
    delivery_times, popmean=10, alternative="greater"
)

two_sided_result = stats.ttest_1samp(
    delivery_times, popmean=10, alternative="two-sided"
)

print("Less p-value:", less_result.pvalue)
print("Greater p-value:", greater_result.pvalue)
print("Two-sided p-value:", two_sided_result.pvalue)

# 9. One-Sample Z-Test

A classic Z-test for a population mean can be used when the population standard deviation is known.

Formula:

\[
z = \frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}
\]

Example:
- hypothesized mean = 100
- known population standard deviation = 15
- sample mean = 105
- sample size = 100

In [ ]:
sample_mean = 105
population_mean = 100
population_std = 15
n = 100

z = (sample_mean - population_mean) / (
    population_std / np.sqrt(n)
)

p_value = 2 * stats.norm.sf(abs(z))

print("Z-statistic:", z)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

# 10. Independent Two-Sample t-Test

Compare the average scores of two independent groups.

### Analogy

- Class A = students taught using Method A
- Class B = different students taught using Method B

The groups are independent.

### Hypotheses

\[
H_0: \mu_A = \mu_B
\]

\[
H_1: \mu_A \neq \mu_B
\]

In [ ]:
group_A = np.array([72, 75, 78, 70, 74, 77, 73, 76, 71, 75])
group_B = np.array([65, 68, 70, 64, 69, 66, 67, 71, 63, 68])

result = stats.ttest_ind(group_A, group_B)

print("Mean A:", group_A.mean())
print("Mean B:", group_B.mean())
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)

# 11. Welch's t-Test

Welch's t-test does not require equal population variances and is often a safer choice when comparing two independent groups whose variances may differ.

In SciPy:

```python
equal_var=False
```

In [ ]:
welch_result = stats.ttest_ind(
    group_A,
    group_B,
    equal_var=False
)

print("Welch t-statistic:", welch_result.statistic)
print("Welch p-value:", welch_result.pvalue)

# 12. Paired t-Test

The same students take a test **before** and **after** training.

Because the observations are linked, the data are paired.

### Analogy

- Independent test → compare two different people/groups
- Paired test → compare the same people before and after

The test works with within-person differences:

\[
d_i = After_i - Before_i
\]

and tests whether the mean difference is zero.

In [ ]:
before = np.array([60, 65, 70, 62, 68, 64, 61, 67, 63, 66])
after = np.array([68, 72, 75, 70, 74, 71, 69, 73, 70, 72])

result = stats.ttest_rel(before, after)

print("Mean before:", before.mean())
print("Mean after:", after.mean())
print("Mean difference:", (after - before).mean())
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)

# 13. Chi-Square Goodness-of-Fit Test

A die is claimed to be fair.

We roll it 600 times. If it is fair, each face should appear about:

\[
600/6 = 100
\]

times.

### Hypotheses

\[
H_0: \text{The die follows the claimed distribution}
\]

\[
H_1: \text{The die does not follow the claimed distribution}
\]

The chi-square statistic compares observed and expected counts:

\[
\chi^2 = \sum \frac{(O-E)^2}{E}
\]

In [ ]:
observed = np.array([120, 95, 80, 105, 110, 90])
expected = np.repeat(100, 6)

result = stats.chisquare(observed, f_exp=expected)

print("Chi-square statistic:", result.statistic)
print("p-value:", result.pvalue)

# 14. Chi-Square Test of Independence

### Problem

Is **gender** associated with **product preference**?

| | Like | Don't Like |
|---|---:|---:|
| Male | 60 | 40 |
| Female | 40 | 60 |

### Hypotheses

\[
H_0: \text{The variables are independent}
\]

\[
H_1: \text{The variables are associated}
\]

In [ ]:
table = np.array([
    [60, 40],
    [40, 60]
])

chi2, p_value, dof, expected = stats.chi2_contingency(table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p_value)

print("\nExpected counts:")
print(expected)

# 15. One-Way ANOVA

Compare the average scores of students using three teaching methods.

ANOVA asks:

> Is the variability **between group means** large relative to the variability **within groups**?

\[
F = \frac{\text{Between-group variability}}
{\text{Within-group variability}}
\]

### Hypotheses

\[
H_0: \mu_A = \mu_B = \mu_C
\]

\[
H_1: \text{At least one mean is different}
\]

In [ ]:
method_A = [70, 72, 71, 69, 73]
method_B = [75, 78, 76, 77, 79]
method_C = [85, 88, 86, 87, 89]

result = stats.f_oneway(method_A, method_B, method_C)

print("F-statistic:", result.statistic)
print("p-value:", result.pvalue)

# 16. Post-Hoc Testing with Tukey HSD

If ANOVA is significant, it tells us:

> **At least one group differs.**

It does not tell us exactly which groups differ.

Tukey's HSD can compare pairs while accounting for multiple comparisons.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

scores = method_A + method_B + method_C
groups = (
    ["A"] * len(method_A)
    + ["B"] * len(method_B)
    + ["C"] * len(method_C)
)

tukey = pairwise_tukeyhsd(
    endog=scores,
    groups=groups,
    alpha=0.05
)

print(tukey)

# 17. Pearson Correlation Hypothesis Test

Suppose we want to know whether study hours and exam score are linearly associated.

### Hypotheses

\[
H_0: \rho = 0
\]

\[
H_1: \rho \neq 0
\]

The Pearson correlation coefficient ranges from -1 to +1.

- +1 → perfect positive linear relationship
- 0 → no linear correlation
- -1 → perfect negative linear relationship

**Important:** correlation does not prove causation.

In [ ]:
study_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
exam_scores = np.array([52, 55, 60, 65, 68, 72, 75, 80, 85, 90])

result = stats.pearsonr(study_hours, exam_scores)

print("Pearson correlation:", result.statistic)
print("p-value:", result.pvalue)

# 18. Regression Hypothesis Testing

Consider:

\[
Salary = \beta_0 + \beta_1 Experience + \epsilon
\]

We can test whether experience has a statistically detectable linear relationship with salary.

### Hypotheses

\[
H_0: \beta_1 = 0
\]

\[
H_1: \beta_1 \neq 0
\]

In the regression summary, inspect the coefficient and its `P>|t|` value.

In [ ]:
experience = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
salary = np.array([35, 38, 42, 45, 50, 55, 60, 65, 70, 75])

X = sm.add_constant(experience)
model = sm.OLS(salary, X).fit()

print(model.summary())

# 19. Mann–Whitney U Test

A non-parametric test for two independent groups.

It can be useful when the assumptions of a standard independent t-test are not appropriate.

### Intuition

The test uses the **ranks** of observations.

> Non-parametric does **not** mean "no assumptions." It still has assumptions and answers a somewhat different question from a t-test.

In [ ]:
group_A = [2, 3, 4, 5, 100]
group_B = [6, 7, 8, 9, 10]

result = stats.mannwhitneyu(
    group_A,
    group_B,
    alternative="two-sided"
)

print("U statistic:", result.statistic)
print("p-value:", result.pvalue)

# 20. Wilcoxon Signed-Rank Test

For paired data, the Wilcoxon signed-rank test is a common non-parametric alternative to the paired t-test when its assumptions are not appropriate.

In [ ]:
before = [50, 55, 60, 52, 58]
after = [55, 60, 65, 59, 63]

result = stats.wilcoxon(before, after)

print("Wilcoxon statistic:", result.statistic)
print("p-value:", result.pvalue)

# 21. Kruskal–Wallis Test

For three or more independent groups, Kruskal–Wallis is a common non-parametric alternative to one-way ANOVA.

Think:

> **ANOVA, but rank-based.**

In [ ]:
A = [10, 12, 15, 14, 13]
B = [20, 22, 19, 21, 23]
C = [15, 17, 16, 18, 14]

result = stats.kruskal(A, B, C)

print("Kruskal-Wallis statistic:", result.statistic)
print("p-value:", result.pvalue)

# 22. Friedman Test

The Friedman test is a non-parametric test for **three or more related/paired conditions**.

Example: the same participants try Treatment A, B, and C.

Think:

> **Friedman ≈ non-parametric repeated-measures ANOVA.**

In [ ]:
treatment_A = [10, 12, 14, 11, 13]
treatment_B = [15, 17, 18, 16, 19]
treatment_C = [20, 21, 23, 22, 24]

result = stats.friedmanchisquare(
    treatment_A,
    treatment_B,
    treatment_C
)

print("Friedman statistic:", result.statistic)
print("p-value:", result.pvalue)

# 23. McNemar's Test — Especially Useful for ML

Suppose two classifiers make predictions on the **same test examples**.

McNemar's test is useful for paired binary classification outcomes.

It focuses on disagreements:

- Model A correct, Model B wrong
- Model A wrong, Model B correct

This is often more appropriate than blindly comparing two accuracy percentages.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

table = np.array([
    [800, 100],
    [150, 50]
])

result = mcnemar(
    table,
    exact=False,
    correction=True
)

print("Statistic:", result.statistic)
print("p-value:", result.pvalue)

# 24. Confidence Intervals

A hypothesis test gives us a decision about a null hypothesis.

A confidence interval gives us a **range of plausible values** for a population parameter under the chosen procedure.

For many standard two-sided tests:

> If the null value is outside the 95% confidence interval, the corresponding test is significant at approximately α = 0.05.

In [ ]:
data = np.array([70, 72, 75, 68, 74, 77, 71, 73, 69, 76])

mean = data.mean()
sem = stats.sem(data)

ci = stats.t.interval(
    confidence=0.95,
    df=len(data) - 1,
    loc=mean,
    scale=sem
)

print("Sample mean:", mean)
print("95% confidence interval:", ci)

# 25. Effect Size

A p-value tells us about evidence against H₀, but it does not tell us how **large or practically important** an effect is.

For two independent groups, one common measure is **Cohen's d**:

\[
d = \frac{\bar{x}_1-\bar{x}_2}{s_{pooled}}
\]

Always consider:

1. statistical significance
2. effect size
3. confidence interval
4. practical/real-world importance

In [ ]:
group_A = np.array([72, 75, 78, 70, 74])
group_B = np.array([65, 68, 70, 64, 69])

n_A = len(group_A)
n_B = len(group_B)

pooled_std = np.sqrt(
    (
        (n_A - 1) * np.var(group_A, ddof=1)
        + (n_B - 1) * np.var(group_B, ddof=1)
    ) / (n_A + n_B - 2)
)

cohens_d = (group_A.mean() - group_B.mean()) / pooled_std

print("Cohen's d:", cohens_d)

# 26. Type I Error

### Analogy: false arrest

Suppose H₀ is actually true, but your test rejects H₀.

You have made a **Type I error**.

This is a **false positive**.

With α = 0.05, a correctly calibrated procedure has a long-run Type I error rate of about 5% under the relevant assumptions.

In [ ]:
np.random.seed(42)

false_positives = 0
experiments = 1000

# H0 is actually true: population mean = 100
for _ in range(experiments):
    sample = np.random.normal(loc=100, scale=10, size=30)
    result = stats.ttest_1samp(sample, 100)

    if result.pvalue < 0.05:
        false_positives += 1

print("False positives:", false_positives)
print("Observed false-positive rate:",
      false_positives / experiments)

# 27. Type II Error and Statistical Power

A **Type II error** happens when H₀ is false, but we fail to reject it.

This is a **false negative**.

Statistical power is:

\[
Power = 1 - \beta
\]

Power asks:

> If a real effect exists, how likely is my experiment to detect it?

A common target is 80% power.

In [ ]:
np.random.seed(42)

experiments = 1000
significant = 0

# H0 says mean = 100, but the true mean is 105
for _ in range(experiments):
    sample = np.random.normal(loc=105, scale=10, size=30)
    result = stats.ttest_1samp(sample, 100)

    if result.pvalue < 0.05:
        significant += 1

estimated_power = significant / experiments

print("Estimated power:", estimated_power)

# 28. Power Analysis

Instead of asking only "How much power do I have?", we can ask:

> **How many observations do I need?**

Power generally increases with:
- larger sample size
- larger effect size
- lower noise
- a less strict alpha threshold (at the cost of greater Type I error risk)

In [ ]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

required_n = analysis.solve_power(
    effect_size=0.5,
    alpha=0.05,
    power=0.80,
    alternative="two-sided"
)

print("Approximate required sample size per group:",
      np.ceil(required_n))

# 29. Multiple Hypothesis Testing

Suppose you test **100 hypotheses** at α = 0.05.

Even if every null hypothesis is true, random variation can produce some p-values below 0.05.

The rough expected number of false positives under idealized independent null tests is:

\[
100 \times 0.05 = 5
\]

This is the **multiple comparisons problem**.

In [ ]:
np.random.seed(42)

p_values = np.random.uniform(0, 1, 100)
significant = p_values < 0.05

print("Number with p < 0.05:", significant.sum())

# 30. Bonferroni Correction

One simple correction is:

\[
\alpha_{adjusted} = \frac{\alpha}{m}
\]

where `m` is the number of tests.

For 10 tests:

\[
0.05/10 = 0.005
\]

Bonferroni is simple and conservative.

In [ ]:
from statsmodels.stats.multitest import multipletests

bonferroni = multipletests(
    p_values,
    alpha=0.05,
    method="bonferroni"
)

print("Reject decisions:")
print(bonferroni[0])

print("\nAdjusted p-values:")
print(bonferroni[1])

# 31. Benjamini–Hochberg / False Discovery Rate

The **False Discovery Rate (FDR)** approach controls the expected proportion of false discoveries among the rejected hypotheses.

A common procedure is **Benjamini–Hochberg**.

Compared with Bonferroni, FDR procedures are often less conservative and useful when many hypotheses are being tested.

In [ ]:
fdr_result = multipletests(
    p_values,
    alpha=0.05,
    method="fdr_bh"
)

print("Reject decisions:")
print(fdr_result[0])

print("\nAdjusted p-values:")
print(fdr_result[1])

# 32. Permutation Test

A permutation test constructs an empirical null distribution by randomly shuffling group labels.

### Analogy

Suppose Group A and Group B labels are meaningless under H₀.

Mix all observations together, randomly shuffle the labels, and repeatedly calculate the difference between groups.

Then ask:

> How often does random rearrangement produce an effect as extreme as the one we actually observed?

That gives an empirical p-value.

In [ ]:
group_A = np.array([10, 12, 14, 15, 13])
group_B = np.array([20, 22, 18, 21, 23])

observed_difference = group_A.mean() - group_B.mean()
combined = np.concatenate([group_A, group_B])

permuted_differences = []

for _ in range(10000):
    shuffled = np.random.permutation(combined)
    new_A = shuffled[:len(group_A)]
    new_B = shuffled[len(group_A):]

    difference = new_A.mean() - new_B.mean()
    permuted_differences.append(difference)

permuted_differences = np.array(permuted_differences)

p_value = np.mean(
    np.abs(permuted_differences) >= abs(observed_difference)
)

print("Observed difference:", observed_difference)
print("Permutation p-value:", p_value)

# 33. Bootstrap

Bootstrap resampling repeatedly samples **with replacement** from the observed data.

### Intuition

We treat the observed sample as a practical approximation to the population and repeatedly create bootstrap samples.

This can help estimate:
- uncertainty
- standard errors
- confidence intervals
- sampling distributions of statistics

### Mental shortcut

**Bootstrap:** "How uncertain is my estimate?"

**Permutation:** "How compatible is the observed group difference with the null idea that labels don't matter?"

In [ ]:
data = np.array([10, 12, 15, 17, 20])

bootstrap_means = []

for _ in range(10000):
    bootstrap_sample = np.random.choice(
        data,
        size=len(data),
        replace=True
    )
    bootstrap_means.append(bootstrap_sample.mean())

bootstrap_means = np.array(bootstrap_means)

ci = np.percentile(bootstrap_means, [2.5, 97.5])

print("Original sample mean:", data.mean())
print("Bootstrap 95% percentile CI:", ci)

# 34. A/B Testing

A/B testing is hypothesis testing applied to product experiments.

### Example

An ecommerce company tests two checkout pages.

**Version A**
- 10,000 visitors
- 500 conversions

**Version B**
- 10,000 visitors
- 550 conversions

Conversion rates:

\[
p_A = 500/10000 = 5\%
\]

\[
p_B = 550/10000 = 5.5\%
\]

The question is:

> Is Version B genuinely better, or could the difference be sampling variation?

In [ ]:
visitors_A = 10000
conversions_A = 500

visitors_B = 10000
conversions_B = 550

rate_A = conversions_A / visitors_A
rate_B = conversions_B / visitors_B

print("Conversion rate A:", rate_A)
print("Conversion rate B:", rate_B)
print("Absolute difference:", rate_B - rate_A)
print("Relative lift:", (rate_B - rate_A) / rate_A)

# 35. Two-Proportion Z-Test

We test:

\[
H_0: p_A = p_B
\]

against:

\[
H_1: p_A \neq p_B
\]

This is a common test for conversion-rate A/B experiments.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

successes = np.array([conversions_A, conversions_B])
samples = np.array([visitors_A, visitors_B])

z_stat, p_value = proportions_ztest(successes, samples)

print("Z-statistic:", z_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject H0: evidence of a difference in conversion rates.")
else:
    print("Fail to reject H0: insufficient evidence of a difference.")

# 36. Machine Learning Model Comparison

Suppose two classification models make predictions on the **same test set**.

A common mistake is:

> Model A accuracy = 91%, Model B accuracy = 93%, therefore B is statistically better.

Not necessarily.

You need to consider:
- paired evaluation
- the metric being compared
- uncertainty
- resampling
- dependence between predictions
- the experimental design

For paired binary predictions, **McNemar's test** can be appropriate.

For AUC, specialized tests such as DeLong's test may be appropriate in suitable settings.

For cross-validation, do not automatically treat folds as independent observations.

In [ ]:
from sklearn.metrics import accuracy_score

y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 1])

pred_A = np.array([1, 0, 0, 1, 0, 1, 0, 1, 1, 0])
pred_B = np.array([1, 1, 1, 1, 0, 1, 0, 0, 1, 1])

accuracy_A = accuracy_score(y_true, pred_A)
accuracy_B = accuracy_score(y_true, pred_B)

print("Model A accuracy:", accuracy_A)
print("Model B accuracy:", accuracy_B)

# 37. Choosing the Right Hypothesis Test

| Situation | Typical test |
|---|---|
| One mean vs known value | One-sample t-test |
| One mean with known population σ | Z-test |
| Two independent means | Independent t-test |
| Two independent means with unequal variances | Welch's t-test |
| Same subjects before/after | Paired t-test |
| 3+ independent means | One-way ANOVA |
| Find which ANOVA groups differ | Tukey HSD |
| One categorical distribution | Chi-square goodness-of-fit |
| Two categorical variables | Chi-square independence |
| Two independent non-parametric groups | Mann–Whitney U |
| Two paired non-parametric groups | Wilcoxon signed-rank |
| 3+ independent non-parametric groups | Kruskal–Wallis |
| 3+ related non-parametric groups | Friedman |
| Paired binary classifier outcomes | McNemar |
| Linear correlation | Pearson correlation test |
| Compare two proportions | Two-proportion z-test |
| Flexible null distribution | Permutation test |
| Estimate uncertainty | Bootstrap |

# 38. The Most Important Workflow

Do **not** start by asking:

> "Which Python function should I use?"

Start with the research question.

### Step 1 — Define the question
Example: Does the new teaching method improve scores?

### Step 2 — Define H₀
\[
H_0: \mu_{new}=\mu_{old}
\]

### Step 3 — Define H₁
\[
H_1: \mu_{new}\neq\mu_{old}
\]

### Step 4 — Choose α
Usually 0.05.

### Step 5 — Understand the data
Ask:
- What type of variable?
- How many groups?
- Independent or paired?
- What assumptions are reasonable?

### Step 6 — Select the test

### Step 7 — Calculate the statistic and p-value

### Step 8 — Make the statistical decision

\[
p < \alpha \Rightarrow Reject H_0
\]

### Step 9 — Report effect size and confidence interval

### Step 10 — Explain the practical meaning

This thought process is more important than memorizing Python functions.

In [ ]:
def interpret_p_value(p_value, alpha=0.05):
    if p_value < alpha:
        return "Reject H0: statistically significant evidence against H0."
    return "Fail to reject H0: insufficient evidence against H0."

print(interpret_p_value(0.03))
print(interpret_p_value(0.20))

# 39. Common Mistakes

### ❌ "p = 0.03 means there is a 3% probability H₀ is true."
**Correct:** p-value is calculated under the assumption that H₀ is true.

### ❌ "p > 0.05 proves H₀."
**Correct:** We fail to reject H₀.

### ❌ "Statistically significant means practically important."
**Correct:** Check effect size and confidence interval.

### ❌ "I'll choose a one-tailed test after seeing the data."
**Correct:** Choose the direction from the research question before analysis.

### ❌ "I ran 100 tests and found p < 0.05, so I found something."
**Correct:** Consider multiple-testing corrections.

### ❌ "I'll use a t-test for everything."
**Correct:** Test choice depends on the question, study design, data type, and assumptions.

# 40. Master Mental Model

Think of yourself as a detective:

```text
Research Question
       ↓
     H₀ / H₁
       ↓
   Collect Data
       ↓
 Measure Evidence
       ↓
 Test Statistic
       ↓
    P-value
       ↓
 ┌─────┴─────┐
 ↓           ↓
Small p    Large p
 ↓           ↓
Reject      Fail to
H₀          reject H₀
 ↓           ↓
Effect Size + Confidence Interval
       ↓
Practical Interpretation
```

## The four ideas to remember

1. **Null hypothesis:** What would I assume if nothing special were happening?
2. **Evidence:** How different is my observation from what H₀ predicts?
3. **P-value:** How surprising would this result be if H₀ were true?
4. **Decision:** Is the evidence strong enough to reject H₀?

> **Never interpret a p-value by itself.**

Good statistical analysis combines:

**research question + appropriate test + assumptions + p-value + effect size + confidence interval + practical meaning.**

# 41. Practice Exercises

Try these yourself before looking up the answer.

## Exercise 1 — One-sample t-test

A company claims average customer wait time is 8 minutes.

```python
wait_times = np.array([
    7.5, 8.2, 7.8, 8.1, 7.4,
    7.9, 8.3, 7.6, 7.7, 7.5,
    8.0, 7.8
])
```

Test whether the average wait time differs from 8 minutes.

---

## Exercise 2 — Paired t-test

Students take a test before and after an AI tutoring program.

```python
before = np.array([55, 60, 62, 58, 65, 61, 57, 59, 63, 56])
after  = np.array([62, 68, 67, 65, 70, 66, 64, 67, 69, 63])
```

Is there evidence that the tutoring program changed scores?

---

## Exercise 3 — A/B testing

Website A:
- 5,000 visitors
- 250 purchases

Website B:
- 5,000 visitors
- 285 purchases

Test whether conversion rates differ.

---

## Exercise 4 — Correlation

```python
study_hours = np.array([2, 3, 4, 5, 6, 7, 8, 9])
scores = np.array([55, 58, 62, 65, 69, 73, 78, 82])
```

Test whether study hours and scores have a statistically significant linear relationship.

---

## Exercise 5 — ML model comparison

Two models classify the same 500 test images.

Model A is correct on 420 images.
Model B is correct on 435 images.

Ask:

> Is comparing 84% vs 87% accuracy enough to conclude that Model B is significantly better?

Think about the **paired predictions** and which statistical test could be appropriate.

# Final Cheat Sheet

- **H₀** → default/no-effect assumption
- **H₁** → alternative claim
- **α** → significance threshold
- **Test statistic** → standardized measure of evidence
- **p-value** → how unusual the observed result is under H₀
- **Reject H₀** → evidence is sufficiently strong under the chosen threshold
- **Fail to reject H₀** → insufficient evidence
- **Type I error** → false positive
- **Type II error** → false negative
- **Power** → probability of detecting a specified real effect
- **Effect size** → how large the effect is
- **Confidence interval** → uncertainty/plausible range from the chosen procedure
- **Multiple testing** → many tests increase opportunities for false positives

## Golden rule

> **Never interpret a p-value by itself.**

Always ask:

1. What was the research question?
2. What are H₀ and H₁?
3. Why was this test chosen?
4. What assumptions does it require?
5. What is the p-value?
6. What is the effect size?
7. What is the confidence interval?
8. Is the effect practically important?
9. Were multiple comparisons made?
10. What does the result mean in the real-world context?